# Notebook 03: Library MSTL (Statsmodels) & Benchmark Comparisons

**Project**: MSTL Implementation and Extensions  
**Authors**: Rup Narayan Jha, Jaypal Malviya, Anuj Kumar, Ravi Kiran

---

## 1. Objectives
In this notebook, we compare our Custom MSTL implementation against established benchmark methods:
1. **Statsmodels MSTL**: The standard Python library implementation (`statsmodels.tsa.seasonal.MSTL`).
2. **Traditional STL**: Single-period LOESS decomposition ($m=24$).
3. **TBATS**: Trigonometric Box-Cox ARMA Trend Seasonal state-space model.

We measure:
- Component fidelity and correlation.
- Wall-clock execution time.
- Residual characteristics.

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('../src'))

from data_processing import load_hourly, extract_experiment_period
from mstl_custom import mstl_decompose
from benchmarks import run_stl, run_mstl_library, run_tbats

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
sns.set_theme(style='whitegrid')

## 2. Load Series

In [ ]:
hourly = load_hourly('../data/vic_elec_hourly.csv')
series = extract_experiment_period(hourly, n_obs=3601)
x = series.values.astype(float)

## 3. Compare Custom MSTL vs Library MSTL

In [ ]:
t0 = time.perf_counter()
custom_res = mstl_decompose(x, periods=[24, 168], iterations=2)
t_custom = time.perf_counter() - t0

lib_res = run_mstl_library(x, periods=(24, 168), iterations=2)

print(f"Custom MSTL runtime:  {t_custom:.3f} s")
print(f"Library MSTL runtime: {lib_res['time_sec']:.3f} s")

trend_diff = np.max(np.abs(custom_res['trend'] - lib_res['trend']))
print(f"Max absolute difference in trend: {trend_diff:.4f} MWh")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(series.index, custom_res['trend'], label='Custom MSTL Trend', color='crimson', lw=1.2)
ax.plot(series.index, lib_res['trend'], label='Statsmodels MSTL Trend', color='navy', lw=1.0, ls='--')
ax.set_title("Trend Comparison: Custom MSTL vs Statsmodels MSTL", fontweight='bold')
ax.set_ylabel("Trend (MWh)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Benchmark: Traditional Single-Period STL ($m=24$)
STL can model only a single period. Notice how omitting the weekly period leaves weekly oscillations in the residuals.

In [ ]:
stl_res = run_stl(x, period=24, seasonal=25)
print(f"STL runtime: {stl_res['time_sec']:.3f} s")
print(f"STL residual std: {np.std(stl_res['remainder']):.2f} MWh")

## 5. Benchmark: TBATS Model
TBATS models seasonal patterns via Fourier terms within an exponential smoothing state-space model. We evaluate it on 500 observations due to its quadratic computational scaling.

In [ ]:
tbats_res = run_tbats(x, seasonal_periods=(24, 168), max_obs=500)
print(f"TBATS runtime (500 obs): {tbats_res['time_sec']:.3f} s")
print(f"TBATS AIC: {tbats_res['aic']:.2f}")

## 6. Comprehensive Method Comparison Table

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Method': 'Custom MSTL',
        'Seasonal Periods': '24, 168',
        'Observations': len(x),
        'Runtime (s)': f"{t_custom:.3f}",
        'Remainder Std (MWh)': f"{np.std(custom_res['remainder']):.2f}",
        'Recon. Error': '< 1e-12'
    },
    {
        'Method': 'Statsmodels MSTL',
        'Seasonal Periods': '24, 168',
        'Observations': len(x),
        'Runtime (s)': f"{lib_res['time_sec']:.3f}",
        'Remainder Std (MWh)': f"{np.std(lib_res['remainder']):.2f}",
        'Recon. Error': '< 1e-12'
    },
    {
        'Method': 'Standard STL',
        'Seasonal Periods': '24 only',
        'Observations': len(x),
        'Runtime (s)': f"{stl_res['time_sec']:.3f}",
        'Remainder Std (MWh)': f"{np.std(stl_res['remainder']):.2f}",
        'Recon. Error': '< 1e-12'
    },
    {
        'Method': 'TBATS',
        'Seasonal Periods': '24, 168',
        'Observations': tbats_res['n_obs'],
        'Runtime (s)': f"{tbats_res['time_sec']:.3f}",
        'Remainder Std (MWh)': f"{np.std(tbats_res['remainder']):.2f}",
        'Recon. Error': 'N/A (State-space)'
    }
]).set_index('Method')

comparison_df